# Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import tqdm
import pandas as pd

from sklearn.metrics import ndcg_score
from scipy.sparse import csr_matrix

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
display(device)

device(type='cuda')

In [ ]:
!wget -nc https://files.grouplens.org/datasets/movielens/ml-100k.zip
!unzip -n 'ml-100k.zip'
!cat ml-100k/README

File ‘ml-100k.zip’ already there; not retrieving.

Archive:  ml-100k.zip
SUMMARY & USAGE LICENSE

MovieLens data sets were collected by the GroupLens Research Project
at the University of Minnesota.
 
This data set consists of:
	* 100,000 ratings (1-5) from 943 users on 1682 movies. 
	* Each user has rated at least 20 movies. 
        * Simple demographic info for the users (age, gender, occupation, zip)

The data was collected through the MovieLens web site
(movielens.umn.edu) during the seven-month period from September 19th, 
1997 through April 22nd, 1998. This data has been cleaned up - users
who had less than 20 ratings or did not have complete demographic
information were removed from this data set. Detailed descriptions of
the data file can be found at the end of this file.

Neither the University of Minnesota nor any of the researchers
involved can guarantee the correctness of the data, its suitability
for any particular purpose, or the validity of results based on the
use of t

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [ ]:
ratings = pd.read_csv("ml-100k/u.data",
                      delimiter="\t",
                      names=["userId", "itemId", "rating", "tstamp"],
                      engine='python')
ratings

,userId,itemId,rating,tstamp
0,196,242,3,881250949
1,186,302,3,891717742
2,22,377,1,878887116
3,244,51,2,880606923
4,166,346,1,886397596
...,...,...,...,...
99995,880,476,3,880175444
99996,716,204,5,879795543
99997,276,1090,1,874795795
99998,13,225,2,882399156


In [ ]:
# Attribution: Google's search AI

from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader

ratings['userId'] = ratings['userId'].astype('category').cat.codes
ratings['itemId'] = ratings['itemId'].astype('category').cat.codes
n_users = len(ratings['userId'].unique())
n_items = len(ratings['itemId'].unique())
# Split the data into training and testing sets
train_df, test_df = train_test_split(ratings, test_size=0.2, random_state=42)

class MovieLensDataset(Dataset):
    def __init__(self, df):
        self.user_ids = torch.tensor(df['userId'].values, dtype=torch.long)
        self.movie_ids = torch.tensor(df['itemId'].values, dtype=torch.long)
        self.ratings = torch.tensor(df['rating'].values, dtype=torch.float)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, idx):
        return self.user_ids[idx], self.movie_ids[idx], self.ratings[idx]

# Create datasets
train_dataset = MovieLensDataset(train_df)
test_dataset = MovieLensDataset(test_df)

# Models

In [ ]:
class MF(nn.Module):
    """ Matrix factorization model simple """
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_embeddings=num_users, embedding_dim=emb_dim)
        self.item_emb = nn.Embedding(num_embeddings=num_items, embedding_dim=emb_dim)

    def forward(self, user, item):
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)
        element_product = torch.dot(user_emb, item_emb)
        return element_product

In [ ]:
# My GMF implementation is designed for use on explicit data rated between 1-5
class GMF(nn.Module):
    """ Generalized Matrix Factorization model simple """
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_embeddings=num_users, embedding_dim=emb_dim)
        self.item_emb = nn.Embedding(num_embeddings=num_items, embedding_dim=emb_dim)
        self.output_layer = nn.Linear(in_features=emb_dim, out_features=1)

    def elementwise_product(self, user, item):
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)
        return user_emb * item_emb

    def forward(self, user, item):
        elementwise_product = self.elementwise_product(user, item)
        prediction = self.output_layer(elementwise_product)
        return prediction

In [ ]:
class MLP(nn.Module):
    """ Multi Layer Perceptron model simple """
    def __init__(self, num_users, num_items, emb_dim):
        super().__init__()
        self.user_emb = nn.Embedding(num_embeddings=num_users, embedding_dim=emb_dim)
        self.item_emb = nn.Embedding(num_embeddings=num_items, embedding_dim=emb_dim)

        input_dim = emb_dim * 2
        fc1_dim = input_dim // 2
        self.fc1 = nn.Linear(in_features=input_dim, out_features=fc1_dim)
        fc2_dim = fc1_dim // 2
        self.fc2 = nn.Linear(in_features=fc1_dim, out_features=fc2_dim)

        self.output_layer = nn.Linear(in_features=fc2_dim, out_features=1)

    def hidden_output(self, user, item):
        user_emb = self.user_emb(user)
        item_emb = self.item_emb(item)

        input = torch.cat((user_emb, item_emb), dim=0)

        fc1 = F.relu(self.fc1(input))
        fc2 = F.relu(self.fc2(fc1))

        return fc2

    def forward(self, user, item):
        hidden_output = self.hidden_output(user, item)
        prediction = self.output_layer(hidden_output)
        return prediction

In [ ]:
class NeuMF(nn.Module):
    """ Neural Matrix Factorization model simple """
    def __init__(self, num_users, num_items, gmf, mlp):
        super().__init__()

        self.gmf = gmf
        self.mlp = mlp

        emb_dim = mlp.output_layer.in_features + gmf.output_layer.in_features

        self.output_layer = nn.Linear(in_features=emb_dim, out_features=1)

        # self.user_emb = nn.Embedding(num_embeddings=num_users, embedding_dim=emb_dim)
        # self.item_emb = nn.Embedding(num_embeddings=num_items, embedding_dim=emb_dim)

        # input_dim = emb_dim * 2
        # fc1_dim = input_dim // 2
        # self.fc1 = nn.Linear(in_features=input_dim, out_features=fc1_dim)
        # fc2_dim = fc1_dim // 2
        # self.fc2 = nn.Linear(in_features=fc1_dim, out_features=fc2_dim)

        # self.output_layer = nn.Linear(in_features=fc2_dim, out_features=1)

    def forward(self, user, item):
        gmf_output = self.gmf.elementwise_product(user, item)
        mlp_output = self.mlp.hidden_output(user, item)

        input = torch.cat((gmf_output, mlp_output), dim=0)

        prediction = self.output_layer(input)
        return prediction



        # user_emb = self.user_emb(user)
        # item_emb = self.item_emb(item)

        # input = torch.cat((user_emb, item_emb), dim=0)

        # fc1 = F.relu(self.fc1(input))
        # fc2 = F.relu(self.fc2(fc1))
        # prediction = self.output_layer(fc2)
        # return prediction

# Train

In [ ]:
def train(model, num_epochs = 10, optimizer = "sgd", loss = "rmse"):
  if optimizer == "sgd":
    opt = torch.optim.SGD(model.parameters())
  elif optimizer == "adam":
    opt = torch.optim.Adam(model.parameters())

  if loss == "mse" or loss == "rmse":
    loss_fn = nn.MSELoss()
  elif loss == "bce":
    loss_fn = nn.BCELoss()

  epoch_train_losses, epoch_val_losses = [], []

  for i in range(num_epochs):
      train_losses, val_losses = [], []
      model.train()
      for uid, mid, rating in tqdm.tqdm(train_dataset):
          xUser = uid.to(device, dtype=torch.long)
          xItem = mid.to(device, dtype=torch.long)
          preds = model(xUser, xItem)
          yRatings = rating.to(device, dtype=torch.float).view(preds.shape)
          loss = loss_fn(preds, yRatings)
          if loss == "rmse":
            loss = torch.sqrt(loss)
          train_losses.append(loss.item())
          opt.zero_grad()
          loss.backward()
          opt.step()
      model.eval()
      for uid, mid, rating in test_dataset: # Compute MSE for test ratings!
          xUser = uid.to(device, dtype=torch.long)
          xItem = mid.to(device, dtype=torch.long)
          preds = model(xUser, xItem)
          yRatings = rating.to(device, dtype=torch.float).view(preds.shape)
          loss = loss_fn(preds, yRatings)
          if loss == "rmse":
            loss = torch.sqrt(loss)
          val_losses.append(loss.item())
      # Start logging
      epoch_train_loss = np.mean(train_losses)
      epoch_val_loss = np.mean(val_losses)
      epoch_train_losses.append(epoch_train_loss)
      epoch_val_losses.append(epoch_val_loss)
      print(f'Epoch: {i}, Train Loss: {epoch_train_loss:0.2f}, Val Loss:{epoch_val_loss:0.2f}')

# Execution

In [ ]:
mdl = MF(n_users, n_items, emb_dim=32)
mdl.to(device)
print(mdl)

train(mdl, num_epochs=4, optimizer="adam")

MF(
  (user_emb): Embedding(943, 32)
  (item_emb): Embedding(1682, 32)
)


100%|██████████| 80000/80000 [01:46<00:00, 753.72it/s]


Epoch: 0, Train Loss: 33.39, Val Loss:24.02


100%|██████████| 80000/80000 [01:41<00:00, 786.33it/s]


Epoch: 1, Train Loss: 12.50, Val Loss:8.95


100%|██████████| 80000/80000 [01:41<00:00, 788.72it/s]


Epoch: 2, Train Loss: 3.89, Val Loss:4.69


100%|██████████| 80000/80000 [01:40<00:00, 793.85it/s]


Epoch: 3, Train Loss: 2.00, Val Loss:3.45


In [ ]:
gmf = GMF(n_users, n_items, emb_dim=32)
gmf.to(device)
print(gmf)

train(gmf, num_epochs=4, optimizer="adam")

GMF(
  (user_emb): Embedding(943, 32)
  (item_emb): Embedding(1682, 32)
  (output_layer): Linear(in_features=32, out_features=1, bias=True)
)


100%|██████████| 80000/80000 [02:02<00:00, 655.25it/s]


Epoch: 0, Train Loss: 1.52, Val Loss:1.29


100%|██████████| 80000/80000 [02:00<00:00, 666.64it/s]


Epoch: 1, Train Loss: 1.25, Val Loss:1.34


100%|██████████| 80000/80000 [01:59<00:00, 666.71it/s]


Epoch: 2, Train Loss: 1.20, Val Loss:1.41


100%|██████████| 80000/80000 [01:59<00:00, 666.95it/s]


Epoch: 3, Train Loss: 1.14, Val Loss:1.47


In [ ]:
mlp = MLP(n_users, n_items, emb_dim=32)
mlp.to(device)
print(mlp)

train(mlp, num_epochs=4, optimizer="adam")

MLP(
  (user_emb): Embedding(943, 32)
  (item_emb): Embedding(1682, 32)
  (fc1): Linear(in_features=64, out_features=32, bias=True)
  (fc2): Linear(in_features=32, out_features=16, bias=True)
  (output_layer): Linear(in_features=16, out_features=1, bias=True)
)


100%|██████████| 80000/80000 [02:35<00:00, 513.60it/s]


Epoch: 0, Train Loss: 1.11, Val Loss:1.00


100%|██████████| 80000/80000 [02:35<00:00, 514.38it/s]


Epoch: 1, Train Loss: 0.93, Val Loss:0.95


100%|██████████| 80000/80000 [02:34<00:00, 516.58it/s]


Epoch: 2, Train Loss: 0.89, Val Loss:0.95


100%|██████████| 80000/80000 [02:34<00:00, 518.45it/s]


Epoch: 3, Train Loss: 0.86, Val Loss:0.93


In [ ]:
neumf = NeuMF(n_users, n_items, gmf, mlp)
neumf.to(device)
print(neumf)

train(neumf, num_epochs=4, optimizer="adam")

NeuMF(
  (gmf): GMF(
    (user_emb): Embedding(943, 32)
    (item_emb): Embedding(1682, 32)
    (output_layer): Linear(in_features=32, out_features=1, bias=True)
  )
  (mlp): MLP(
    (user_emb): Embedding(943, 32)
    (item_emb): Embedding(1682, 32)
    (fc1): Linear(in_features=64, out_features=32, bias=True)
    (fc2): Linear(in_features=32, out_features=16, bias=True)
    (output_layer): Linear(in_features=16, out_features=1, bias=True)
  )
  (output_layer): Linear(in_features=48, out_features=1, bias=True)
)


100%|██████████| 80000/80000 [02:54<00:00, 459.04it/s]


Epoch: 0, Train Loss: 0.80, Val Loss:1.07


100%|██████████| 80000/80000 [02:52<00:00, 462.65it/s]


Epoch: 1, Train Loss: 0.70, Val Loss:1.11


100%|██████████| 80000/80000 [02:53<00:00, 461.95it/s]


Epoch: 2, Train Loss: 0.65, Val Loss:1.15


100%|██████████| 80000/80000 [02:52<00:00, 463.21it/s]


Epoch: 3, Train Loss: 0.60, Val Loss:1.19


# Evaluation

## Metrics

In [ ]:
def calculate_ndcg(predictions, actual_ratings, k=10, n_users=n_users, n_items=n_items):
  """Calculates NDCG score using a sparse matrix representation."""
  # Create a sparse matrix for actual ratings
  row = test_df['userId'].values
  col = test_df['itemId'].values
  data = actual_ratings.cpu().numpy()  # Convert to NumPy array
  actual_ratings_matrix = csr_matrix((data, (row, col)), shape=(n_users, n_items))

  # Create a sparse matrix for predictions
  predictions_matrix = csr_matrix((predictions.cpu().numpy(), (row, col)), shape=(n_users, n_items))

  # Convert sparse matrices to dense arrays before passing to ndcg_score
  actual_ratings_array = actual_ratings_matrix.toarray()
  predictions_array = predictions_matrix.toarray()

  # Calculate NDCG score using dense arrays
  score = ndcg_score(actual_ratings_array, predictions_array, k=k)
  return score

def calculate_rmse(predictions, actual_ratings):
  diff = predictions - actual_ratings
  squared_diff = diff ** 2
  mean_squared_diff = torch.mean(squared_diff)
  rmse = torch.sqrt(mean_squared_diff).item()
  return rmse

## Results

I have completed full evaluations for all 4 designated algorithms, thereby accomplishing the enhanced version of the assignment

In [ ]:
algorithms: list[nn.Module] = {
    "MF": mdl,
    "GMF": gmf,
    "MLP": mlp,
    "NeuMF": neumf,
}

In [ ]:
results = {}

for algo_name, algorithm in algorithms.items():
  algorithm.to(device)
  algorithm.eval()

  predictions = []
  ratings = []

  with torch.no_grad():
    for uid, mid, rating in tqdm.tqdm(test_dataset):
      user = uid.to(device, dtype=torch.long)
      item = mid.to(device, dtype=torch.long)

      prediction = algorithm(user, item)

      predictions.append(prediction.item())
      ratings.append(rating.item())

  predictions = torch.tensor(predictions)
  ratings = torch.tensor(ratings)

  rmse = calculate_rmse(predictions, ratings)
  ndcg = calculate_ndcg(predictions, ratings)

  results[algo_name] = {
      "RMSE": rmse,
      "NDCG@10": ndcg
  }

100%|██████████| 20000/20000 [00:08<00:00, 2285.72it/s]


In [ ]:
pd.DataFrame.from_dict(results, orient='index')

,RMSE,NDCG@10
MF,1.858637,0.832731
GMF,1.231687,0.843034
MLP,1.009931,0.905790
NeuMF,1.090412,0.885681
